# ============================================================
# Cell 1: Imports and Global Configuration
# ============================================================

In [30]:

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os, sys, json, time
from scipy.spatial import cKDTree
from tqdm.auto import tqdm
from numba import jit, njit, prange
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_style("whitegrid")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ===================== Physical Constants =====================
kB_eV = 8.617333262145e-5          # Boltzmann constant (eV/K)
k_coulomb = 14.399645              # 1/(4*pi*epsilon0) in eV·Å/e^2
NA = 6.02214076e23                 # Avogadro's number

# ===================== Simulation Settings =====================
# Temperature stages and number of steps per stage
TEMPERATURE_STAGES = [4000, 3000, 2000, 1000, 300]   # K
STEPS_PER_STAGE = 1000
TOTAL_STEPS = sum([STEPS_PER_STAGE] * len(TEMPERATURE_STAGES))

# Box and density
BOX_LENGTH = 30.0                  # Å, cubic box
TARGET_DENSITY = 2.2               # g/cm³
BOX_VOLUME = BOX_LENGTH**3

# Monte Carlo move parameters
DISPLACEMENT_MAX = 0.1             # Å, max atom displacement
VOLUME_MOVE_SCALE = 0.01           # relative volume change scale
P_TARGET = 0.0                     # target pressure (GPa) for NPT (set to 0 for ambient)

# Wolf summation parameters
WOLF_ALPHA = 0.2                   # Å⁻¹
COULOMB_CUTOFF = 10.0              # Å

# Buckingham + Coulomb cutoff (same for all short-range)
CUTOFF = 10.0
SKIN = 1.0                         # skin for Verlet list
NEIGHBOR_UPDATE_FREQ = 25

# Trajectory output frequency
TRAJ_OUTPUT_FREQ = 10

# Random generator
SEED = 42
rng = np.random.default_rng(SEED)

print("Global configuration loaded.")
print(f"Temperature schedule: {TEMPERATURE_STAGES} K")
print(f"Total MC steps: {TOTAL_STEPS}")
print(f"Box: {BOX_LENGTH}×{BOX_LENGTH}×{BOX_LENGTH} Å³")
print(f"Wolf alpha = {WOLF_ALPHA}, cutoff = {COULOMB_CUTOFF} Å")

Global configuration loaded.
Temperature schedule: [4000, 3000, 2000, 1000, 300] K
Total MC steps: 5000
Box: 30.0×30.0×30.0 Å³
Wolf alpha = 0.2, cutoff = 10.0 Å


# ============================================================
# Cell 2: Parse Ternary.txt (input structure)
# ============================================================

In [31]:
# ============================================================
# Cell 2 (continued): Validate and wrap coordinates into box
# ============================================================

# Check if any coordinates are outside [0, BOX_LENGTH]
print(f"\nOriginal coordinate ranges:")
print(f"  x: [{coords[:,0].min():.3f}, {coords[:,0].max():.3f}]")
print(f"  y: [{coords[:,1].min():.3f}, {coords[:,1].max():.3f}]")
print(f"  z: [{coords[:,2].min():.3f}, {coords[:,2].max():.3f}]")

# Check for out-of-box atoms
x_out = (coords[:,0] < 0) | (coords[:,0] >= BOX_LENGTH)
y_out = (coords[:,1] < 0) | (coords[:,1] >= BOX_LENGTH)
z_out = (coords[:,2] < 0) | (coords[:,2] >= BOX_LENGTH)
any_out = np.any(x_out | y_out | z_out)

if any_out:
    n_out = np.sum(x_out | y_out | z_out)
    print(f"Warning: {n_out} atoms have coordinates outside the box [0, {BOX_LENGTH}]")
    print("Wrapping coordinates into the box...")
    # Wrap using periodicity
    coords = coords % BOX_LENGTH

# Verify after wrapping
print(f"\nCorrected coordinate ranges:")
print(f"  x: [{coords[:,0].min():.3f}, {coords[:,0].max():.3f}]")
print(f"  y: [{coords[:,1].min():.3f}, {coords[:,1].max():.3f}]")
print(f"  z: [{coords[:,2].min():.3f}, {coords[:,2].max():.3f}]")

# Assert all in box
assert np.all(coords >= 0) and np.all(coords < BOX_LENGTH), "Coordinate wrapping failed!"
print("✓ All coordinates now within simulation box.")


Original coordinate ranges:
  x: [0.005, 32.914]
  y: [0.035, 32.917]
  z: [0.008, 32.909]
Wrapping coordinates into the box...

Corrected coordinate ranges:
  x: [0.005, 30.000]
  y: [0.010, 29.991]
  z: [0.008, 29.977]
✓ All coordinates now within simulation box.


# ============================================================
# Cell 3: Force-field parameter loading (Buckingham)
# ============================================================

In [32]:

def load_buckingham_parameters(param_file=None):
    """
    Load Buckingham parameters for the glass system.
    If a JSON file is provided, read from it; otherwise use defaults.

    Returns
    -------
    buck_params : dict
        Keys are ordered pairs (elem1, elem2) (sorted alphabetically, e.g., ('O','Si')).
        Values are dicts with 'A' (eV), 'rho' (Å), 'C' (eV·Å^6).
    """
    if param_file is not None and os.path.isfile(param_file):
        with open(param_file, 'r') as f:
            buck_params = json.load(f)
            # Convert string keys to tuples if needed (assumes format like "Si-O": {...})
            return {tuple(k.split('-')): v for k, v in buck_params.items()}
    
    # Default parameters (literature values)
    # Sources: Matsui (1994) for Si-O, Ca-O, O-O; adapted for P-O
    default_params = {
        ('Si', 'O'): {'A': 1283.907,   'rho': 0.3205, 'C': 10.44},
        ('Ca', 'O'): {'A': 1272.7,     'rho': 0.2985, 'C': 0.0},
        ('P', 'O') : {'A': 900.0,      'rho': 0.33,   'C': 10.0},   # typical phosphate glass
        ('O', 'O') : {'A': 22764.0,    'rho': 0.149,  'C': 27.88}
    }
    return default_params

# Load parameters
buck_params = load_buckingham_parameters()

print("Buckingham parameters loaded:")
for pair, params in buck_params.items():
    print(f"  {pair[0]}-{pair[1]}: A={params['A']:.2f} eV, rho={params['rho']:.4f} Å, C={params['C']:.2f} eV·Å⁶")

Buckingham parameters loaded:
  Si-O: A=1283.91 eV, rho=0.3205 Å, C=10.44 eV·Å⁶
  Ca-O: A=1272.70 eV, rho=0.2985 Å, C=0.00 eV·Å⁶
  P-O: A=900.00 eV, rho=0.3300 Å, C=10.00 eV·Å⁶
  O-O: A=22764.00 eV, rho=0.1490 Å, C=27.88 eV·Å⁶


# ============================================================
# Cell 4: Charge assignment
# ============================================================

In [33]:

# Partial charges (e)
charge_dict = {
    'Si': 2.4,
    'Ca': 1.2,
    'P' : 3.0,
    'O' : -1.2
}

# Create charge array (N,) in the same order as atoms
charges = np.array([charge_dict[sym] for sym in atom_symbols], dtype=np.float64)

total_charge = np.sum(charges)
print("Charge assignment:")
for el, ch in charge_dict.items():
    count = atom_symbols.count(el)
    if count > 0:
        print(f"  {el}: {ch:+.1f} e × {count} atoms = {ch*count:+.2f} e")
print(f"Total system charge: {total_charge:.6f} e")
assert abs(total_charge) < 1e-6, "System is not charge-neutral! Check charges."

Charge assignment:
  Si: +2.4 e × 420 atoms = +1008.00 e
  Ca: +1.2 e × 252 atoms = +302.40 e
  P: +3.0 e × 56 atoms = +168.00 e
  O: -1.2 e × 1232 atoms = -1478.40 e
Total system charge: 0.000000 e


# ============================================================
# Cell 5: Pairwise parameter matrices and Verlet neighbor list
# ============================================================

In [34]:
# ============================================================
# Cell 5 (نسخه نهایی): Pairwise matrices and Verlet neighbor list
# ============================================================
from scipy.spatial import cKDTree

# --- Build type-indexed parameter matrices ---
num_types = len(type_map)
A_mat = np.zeros((num_types, num_types))
rho_mat = np.zeros((num_types, num_types))
C_mat = np.zeros((num_types, num_types))

for (el1, el2), params in buck_params.items():
    t1 = type_map[el1]
    t2 = type_map[el2]
    A_mat[t1, t2] = params['A']
    rho_mat[t1, t2] = params['rho']
    C_mat[t1, t2] = params['C']
    if t1 != t2:
        A_mat[t2, t1] = params['A']
        rho_mat[t2, t1] = params['rho']
        C_mat[t2, t1] = params['C']

if np.isnan(A_mat).any():
    missing = [(i, j) for i in range(num_types) for j in range(num_types) if np.isnan(A_mat[i, j])]
    raise ValueError(f"Missing Buckingham parameters for type pairs: {missing}")

print("Pairwise Buckingham matrices built.")
print("Type indices:", type_map)

# --- Neighbor list functions ---
def build_neighbor_list(coords, box_size, cutoff, skin):
    """
    Build Verlet neighbor list using cKDTree with PBC.
    
    Returns
    -------
    pairs : ndarray
        Array of index pairs (i, j) within cutoff+skin.
    wrapped_coords : ndarray
        Coordinates wrapped into the primary box.
    """
    if np.isscalar(box_size):
        boxsize = np.array([box_size, box_size, box_size], dtype=np.float64)
    else:
        boxsize = np.asarray(box_size, dtype=np.float64)
    
    box_L = boxsize[0]
    
    # Wrap coordinates into [0, box_L)
    wrapped_coords = coords % box_L
    
    tree = cKDTree(wrapped_coords, boxsize=boxsize)
    pairs = tree.query_pairs(cutoff + skin, output_type='ndarray')
    
    return pairs, wrapped_coords

def update_adjacency(pairs, n_atoms):
    """
    Convert pair list to adjacency list.
    """
    adjacency = [set() for _ in range(n_atoms)]
    for i, j in pairs:
        adjacency[i].add(j)
        adjacency[j].add(i)
    adjacency = [np.array(sorted(list(neighbors)), dtype=np.int32) for neighbors in adjacency]
    return adjacency

# Build initial neighbor list
neighbor_pairs, coords_wrapped = build_neighbor_list(coords, BOX_LENGTH, CUTOFF, SKIN)
coords = coords_wrapped  # replace with wrapped coords
neighbor_adj = update_adjacency(neighbor_pairs, n_atoms)

print(f"Initial neighbor list: {neighbor_pairs.shape[0]} pairs (cutoff+skin={CUTOFF+SKIN:.2f} Å)")

Pairwise Buckingham matrices built.
Type indices: {'Ca': 0, 'O': 1, 'P': 2, 'Si': 3}
Initial neighbor list: 406368 pairs (cutoff+skin=11.00 Å)


# ============================================================
# Cell 6: Total energy calculation (Buckingham + Wolf) – Numba JIT
# ============================================================

In [35]:
# ============================================================
# Cell 6: Total energy calculation (Buckingham + Wolf) – corrected
# ============================================================
from numba import njit
from math import erfc, exp, sqrt, pi

@njit(parallel=False, fastmath=True)
def compute_total_energy(coords, charges, type_indices, A_mat, rho_mat, C_mat,
                         neighbor_pairs, box_size, cutoff, wolf_alpha):
    """
    Compute total potential energy (Buckingham + Wolf) for all pairs in neighbor list.
    """
    n_pairs = neighbor_pairs.shape[0]
    energy = 0.0
    # Precompute Wolf shift constant
    wolf_shift = erfc(wolf_alpha * cutoff) / cutoff
    
    for p in range(n_pairs):
        i = neighbor_pairs[p, 0]
        j = neighbor_pairs[p, 1]
        
        # Displacement vector with minimum image
        dx = coords[i, 0] - coords[j, 0]
        dy = coords[i, 1] - coords[j, 1]
        dz = coords[i, 2] - coords[j, 2]
        
        # Minimum image convention
        dx = dx - box_size * round(dx / box_size)
        dy = dy - box_size * round(dy / box_size)
        dz = dz - box_size * round(dz / box_size)
        
        r_sq = dx*dx + dy*dy + dz*dz
        r = sqrt(r_sq)
        
        # Skip if beyond cutoff or self-interaction
        if r >= cutoff or r < 1e-10:
            continue
        
        # Get pair type indices
        ti = type_indices[i]
        tj = type_indices[j]
        
        A = A_mat[ti, tj]
        rho_val = rho_mat[ti, tj]
        C = C_mat[ti, tj]
        
        # Buckingham term
        buck = 0.0
        if rho_val > 1e-10 and A > 0.0:
            buck = A * exp(-r / rho_val)
        if r > 1e-10:
            buck -= C / (r**6)
        
        # Coulomb Wolf term
        qiqj = charges[i] * charges[j]
        coul = qiqj * (erfc(wolf_alpha * r) / r - wolf_shift)
        
        energy += buck + coul
    
    return energy

# Recompute with safe function
U_pair = compute_total_energy(coords, charges, type_indices, A_mat, rho_mat, C_mat,
                              neighbor_pairs, BOX_LENGTH, CUTOFF, WOLF_ALPHA)

U_self = - (WOLF_ALPHA / sqrt(pi)) * np.sum(charges**2)
U_total = U_pair + U_self

print(f"Pair energy: {U_pair:.6f} eV")
print(f"Self-energy (Wolf): {U_self:.6f} eV")
print(f"Total potential energy: {U_total:.6f} eV")

# Sanity check
if np.isnan(U_total) or np.isinf(U_total):
    print("ERROR: Energy is NaN or Inf! Check parameters and coordinates.")
else:
    print("✓ Energy calculation successful.")

Pair energy: -898152.491624 eV
Self-energy (Wolf): -570.977913 eV
Total potential energy: -898723.469536 eV
✓ Energy calculation successful.


# ============================================================
# Cell 7: Local energy of a single atom (for displacement moves)
# ============================================================

In [36]:
# ============================================================
# Cell 7: Local energy of a single atom – corrected
# ============================================================
from numba import njit
from math import erfc, exp, sqrt

@njit(fastmath=True)
def compute_atom_local_energy(atom_idx, coords, charges, type_indices,
                              A_mat, rho_mat, C_mat, box_size, cutoff, wolf_alpha,
                              neighbor_list):
    """
    Energy contribution of one atom with its neighbors.
    Returns U_i = sum_{j in neighbors} 0.5 * U_ij
    """
    energy = 0.0
    wolf_shift = erfc(wolf_alpha * cutoff) / cutoff
    n_neigh = len(neighbor_list[atom_idx])
    
    for k in range(n_neigh):
        j = neighbor_list[atom_idx][k]
        
        # Skip self-interaction
        if j == atom_idx:
            continue
        
        # Displacement vector with PBC
        dx = coords[atom_idx, 0] - coords[j, 0]
        dy = coords[atom_idx, 1] - coords[j, 1]
        dz = coords[atom_idx, 2] - coords[j, 2]
        
        # Minimum image convention
        dx = dx - box_size * round(dx / box_size)
        dy = dy - box_size * round(dy / box_size)
        dz = dz - box_size * round(dz / box_size)
        
        r_sq = dx*dx + dy*dy + dz*dz
        r = sqrt(r_sq)
        
        # Skip if distance is effectively zero
        if r < 1e-10:
            continue
        
        # Skip if beyond cutoff
        if r >= cutoff:
            continue
        
        ti = type_indices[atom_idx]
        tj = type_indices[j]
        A = A_mat[ti, tj]
        rho_val = rho_mat[ti, tj]
        C = C_mat[ti, tj]
        
        # Buckingham term
        buck = 0.0
        if rho_val > 1e-10 and A > 0.0:
            buck = A * exp(-r / rho_val)
        if r > 1e-10:
            buck -= C / (r**6)
        
        # Coulomb Wolf term
        qiqj = charges[atom_idx] * charges[j]
        coul = qiqj * (erfc(wolf_alpha * r) / r - wolf_shift)
        
        energy += buck + coul
    
    # Return half to avoid double counting
    return energy * 0.5

# Test: sum of local energies should equal U_pair
print("Computing local energies...")
local_energies = np.array([compute_atom_local_energy(i, coords, charges, type_indices,
                                    A_mat, rho_mat, C_mat, BOX_LENGTH, CUTOFF, WOLF_ALPHA,
                                    neighbor_adj) for i in range(n_atoms)])
sum_local = np.sum(local_energies)
print(f"Sum of local energies: {sum_local:.6f} eV")
print(f"Pair energy from full calc: {U_pair:.6f} eV (difference {abs(sum_local-U_pair):.2e})")

# Check for NaN or Inf
if np.isnan(sum_local) or np.isinf(sum_local):
    print("WARNING: NaN or Inf detected in local energies!")
    print(f"Number of NaN: {np.sum(np.isnan(local_energies))}, Inf: {np.sum(np.isinf(local_energies))}")
else:
    print("✓ Local energy calculation successful.")

Computing local energies...
Sum of local energies: -898152.491624 eV
Pair energy from full calc: -898152.491624 eV (difference 6.47e-08)
✓ Local energy calculation successful.


# ============================================================
# Cell 8: Atom displacement Monte Carlo move
# ============================================================

In [37]:
# ============================================================
# Cell 8: Atom displacement Monte Carlo move
# ============================================================
import numpy as np

def attempt_displacement(coords, charges, type_indices, neighbor_adj,
                         A_mat, rho_mat, C_mat, box_size, cutoff, wolf_alpha,
                         max_disp, temperature, energy_pair):
    """
    Attempt a single atom displacement, return updated coords, new pair energy, and whether accepted.
    """
    n_atoms = coords.shape[0]
    i = rng.integers(0, n_atoms)
    old_pos = coords[i].copy()
    
    # Propose displacement
    delta = rng.uniform(-max_disp, max_disp, size=3)
    new_pos = old_pos + delta
    
    # Compute old local energy
    old_local = compute_atom_local_energy(i, coords, charges, type_indices,
                                          A_mat, rho_mat, C_mat, box_size, cutoff, wolf_alpha,
                                          neighbor_adj)
    
    # Temporarily move atom
    coords[i] = new_pos
    new_local = compute_atom_local_energy(i, coords, charges, type_indices,
                                          A_mat, rho_mat, C_mat, box_size, cutoff, wolf_alpha,
                                          neighbor_adj)
    
    delta_e = new_local - old_local
    
    # Metropolis criterion
    accepted = False
    if delta_e <= 0.0:
        accepted = True
    else:
        beta = 1.0 / (kB_eV * temperature)
        prob = exp(-beta * delta_e)
        if rng.random() < prob:
            accepted = True
        else:
            # Reject, restore position
            coords[i] = old_pos
    
    if accepted:
        energy_pair += delta_e
        
    return coords, energy_pair, accepted

# ============================================================
# Cell 9: Volume move for NPT (isotropic)
# ============================================================

In [38]:
# ============================================================
# Cell 9: Volume move for NPT – corrected unpacking
# ============================================================
from math import exp, log

def attempt_volume_move(coords, charges, type_indices,
                        A_mat, rho_mat, C_mat,
                        neighbor_pairs, neighbor_adj,
                        box_length, cutoff, wolf_alpha, skin,
                        vol_scale, temperature, pressure, energy_pair):
    """
    Attempt isotropic volume change.
    Returns updated coords, box_length, neighbor_pairs/adj, energy_pair, and accepted flag.
    """
    n_atoms = coords.shape[0]
    old_box = box_length
    
    # Propose volume change
    delta_vol = vol_scale * (2.0 * rng.random() - 1.0)
    new_box = old_box * (1.0 + delta_vol)
    if new_box <= 0:
        return coords, box_length, neighbor_pairs, neighbor_adj, energy_pair, False
    
    scale_factor = new_box / old_box
    new_coords = coords * scale_factor
    
    # Build temporary neighbor list for new coordinates
    # build_neighbor_list returns (pairs, wrapped_coords)
    temp_pairs, temp_wrapped_coords = build_neighbor_list(new_coords, new_box, cutoff, skin)
    temp_adj = update_adjacency(temp_pairs, n_atoms)
    
    # Use wrapped coordinates for energy calculation
    U_new_pair = compute_total_energy(
        temp_wrapped_coords, charges, type_indices,
        A_mat, rho_mat, C_mat,
        temp_pairs, new_box, cutoff, wolf_alpha
    )
    
    U_old = energy_pair + U_self
    U_new = U_new_pair + U_self
    delta_U = U_new - U_old
    
    # Pressure-volume work (convert GPa → eV/Å³)
    pressure_eV_A3 = pressure * 0.006241509
    vol_old = old_box ** 3
    vol_new = new_box ** 3
    delta_V = vol_new - vol_old
    
    # NPT acceptance weight
    w = delta_U + pressure_eV_A3 * delta_V - n_atoms * kB_eV * temperature * log(vol_new / vol_old)
    
    accepted = False
    if w <= 0:
        accepted = True
    else:
        beta = 1.0 / (kB_eV * temperature)
        if rng.random() < exp(-beta * w):
            accepted = True
    
    if accepted:
        return temp_wrapped_coords, new_box, temp_pairs, temp_adj, U_new_pair, True
    else:
        return coords, box_length, neighbor_pairs, neighbor_adj, energy_pair, False

# ============================================================
# Cell 10: Annealing loop, energy logging, trajectory writer
# ============================================================

In [39]:
# ============================================================
# Cell 10: Annealing loop, energy logging, trajectory writer
# ============================================================
from tqdm.notebook import tqdm
from math import exp, log

# ===================== Initialize logs =====================
n_stages = len(TEMPERATURE_STAGES)
total_steps = n_stages * STEPS_PER_STAGE

energy_log = np.zeros(total_steps)
volume_log = np.zeros(total_steps)
accepted_displ = 0
accepted_vol = 0
attempts_displ = 0
attempts_vol = 0

# ===================== Open trajectory file =====================
traj_file = open("trajectory.xyz", "w")

# ===================== Initialize current state =====================
current_box = BOX_LENGTH
current_energy_pair = U_pair
current_coords = coords.copy()
current_neighbor_pairs = neighbor_pairs.copy()
current_neighbor_adj = [adj.copy() for adj in neighbor_adj]

step_counter = 0
volume_move_prob = 0.01  # 1% volume moves
neighbor_update_counter = 0

print("=" * 60)
print("Starting Monte Carlo NPT Annealing")
print("=" * 60)
print(f"Temperature stages: {TEMPERATURE_STAGES}")
print(f"Steps per stage: {STEPS_PER_STAGE}")
print(f"Total steps: {total_steps}")
print(f"Volume move probability: {volume_move_prob*100:.1f}%")
print(f"Initial energy: {U_total:.3f} eV")
print(f"Initial box: {current_box:.3f} Å")
print("=" * 60)

# ===================== Main annealing loop =====================
start_time = time.time()

for stage, T in enumerate(TEMPERATURE_STAGES):
    print(f"\n{'─'*50}")
    print(f"Stage {stage+1}/{n_stages}: Temperature = {T} K")
    print(f"{'─'*50}")
    
    # Adjust max displacement for lower temperatures
    if T >= 2000:
        max_disp = 0.15  # Å
    elif T >= 1000:
        max_disp = 0.10
    else:
        max_disp = 0.05
    
    # Stage-specific counters
    stage_accept_displ = 0
    stage_attempt_displ = 0
    stage_accept_vol = 0
    stage_attempt_vol = 0
    
    # Progress bar
    pbar = tqdm(range(STEPS_PER_STAGE), desc=f"T={T} K", leave=False)
    
    for s in pbar:
        
        # --- Decide move type ---
        if rng.random() < volume_move_prob:
            # ==========================================
            # Volume move
            # ==========================================
            stage_attempt_vol += 1
            attempts_vol += 1
            
            coords_new, box_new, neigh_pairs_new, neigh_adj_new, en_pair_new, accepted = \
                attempt_volume_move(
                    current_coords, charges, type_indices,
                    A_mat, rho_mat, C_mat,
                    current_neighbor_pairs, current_neighbor_adj,
                    current_box, CUTOFF, WOLF_ALPHA, SKIN,
                    VOLUME_MOVE_SCALE, T, P_TARGET, current_energy_pair
                )
            
            if accepted:
                current_coords = coords_new
                current_box = box_new
                current_neighbor_pairs = neigh_pairs_new
                current_neighbor_adj = neigh_adj_new
                current_energy_pair = en_pair_new
                accepted_vol += 1
                stage_accept_vol += 1
                
        else:
            # ==========================================
            # Displacement move
            # ==========================================
            stage_attempt_displ += 1
            attempts_displ += 1
            
            current_coords, current_energy_pair, accepted = \
                attempt_displacement(
                    current_coords, charges, type_indices,
                    current_neighbor_adj,
                    A_mat, rho_mat, C_mat,
                    current_box, CUTOFF, WOLF_ALPHA,
                    max_disp, T, current_energy_pair
                )
            
            if accepted:
                accepted_displ += 1
                stage_accept_displ += 1
        
        # --- Periodic neighbor list update ---
        if (step_counter + 1) % NEIGHBOR_UPDATE_FREQ == 0:
            current_neighbor_pairs, _ = build_neighbor_list(
                current_coords, current_box, CUTOFF, SKIN
            )
            current_neighbor_adj = update_adjacency(current_neighbor_pairs, n_atoms)
            neighbor_update_counter += 1
        
        # --- Record observables ---
        energy_log[step_counter] = current_energy_pair + U_self
        volume_log[step_counter] = current_box ** 3
        
        # --- Write trajectory every TRAJ_OUTPUT_FREQ steps ---
        if (step_counter + 1) % TRAJ_OUTPUT_FREQ == 0:
            traj_file.write(f"{n_atoms}\n")
            traj_file.write(f"Step {step_counter+1}, T={T} K, Box={current_box:.4f} Å\n")
            for sym, pos in zip(atom_symbols, current_coords):
                traj_file.write(f"{sym:2s} {pos[0]:10.6f} {pos[1]:10.6f} {pos[2]:10.6f}\n")
        
        # --- Update progress bar ---
        if (step_counter + 1) % 100 == 0:
            current_total_energy = current_energy_pair + U_self
            pbar.set_postfix({
                'Energy': f'{current_total_energy:.2f}',
                'Box': f'{current_box:.2f}',
                'Acc_D': f'{stage_accept_displ/max(1,stage_attempt_displ)*100:.1f}%',
                'Acc_V': f'{stage_accept_vol/max(1,stage_attempt_vol)*100:.1f}%'
            })
        
        step_counter += 1
    
    # --- Stage summary ---
    stage_disp_rate = stage_accept_displ / max(1, stage_attempt_displ) * 100
    stage_vol_rate = stage_accept_vol / max(1, stage_attempt_vol) * 100
    stage_avg_energy = np.mean(energy_log[stage*STEPS_PER_STAGE:(stage+1)*STEPS_PER_STAGE])
    stage_avg_volume = np.mean(volume_log[stage*STEPS_PER_STAGE:(stage+1)*STEPS_PER_STAGE])
    
    print(f"\n  Stage {stage+1} summary:")
    print(f"    Energy (avg): {stage_avg_energy:.3f} eV")
    print(f"    Volume (avg): {stage_avg_volume:.1f} Å³")
    print(f"    Displacement acceptance: {stage_accept_displ}/{stage_attempt_displ} = {stage_disp_rate:.1f}%")
    print(f"    Volume acceptance: {stage_accept_vol}/{stage_attempt_vol} = {stage_vol_rate:.1f}%")

# ===================== Close trajectory file =====================
traj_file.close()

# ===================== Final summary =====================
end_time = time.time()
elapsed_time = end_time - start_time

total_disp_rate = accepted_displ / max(1, attempts_displ) * 100
total_vol_rate = accepted_vol / max(1, attempts_vol) * 100

print("\n" + "=" * 60)
print("ANNEALING COMPLETE")
print("=" * 60)
print(f"Total wall time: {elapsed_time:.1f} seconds")
print(f"Time per step: {elapsed_time/total_steps*1000:.2f} ms")
print(f"Neighbor list updates: {neighbor_update_counter}")
print()
print(f"Displacement moves:")
print(f"  Accepted: {accepted_displ}/{attempts_displ} ({total_disp_rate:.1f}%)")
print(f"Volume moves:")
print(f"  Accepted: {accepted_vol}/{attempts_vol} ({total_vol_rate:.1f}%)")
print()
print(f"Final configuration:")
print(f"  Box length: {current_box:.4f} Å")
print(f"  Box volume: {current_box**3:.2f} Å³")
print(f"  Potential energy: {energy_log[-1]:.3f} eV")
print(f"  Energy per atom: {energy_log[-1]/n_atoms:.4f} eV/atom")
print()

# Save final coordinates
print("Saving final coordinates to final_structure.xyz...")
with open("final_structure.xyz", "w") as f:
    f.write(f"{n_atoms}\n")
    f.write(f"Final structure after {total_steps} MC steps, T={TEMPERATURE_STAGES[-1]} K, Box={current_box:.4f} Å\n")
    for sym, pos in zip(atom_symbols, current_coords):
        f.write(f"{sym:2s} {pos[0]:10.6f} {pos[1]:10.6f} {pos[2]:10.6f}\n")
print("Done!")

# Update global variables for subsequent analysis cells
coords = current_coords
box_length = current_box
U_pair = current_energy_pair
neighbor_pairs = current_neighbor_pairs
neighbor_adj = current_neighbor_adj

Starting Monte Carlo NPT Annealing
Temperature stages: [4000, 3000, 2000, 1000, 300]
Steps per stage: 1000
Total steps: 5000
Volume move probability: 1.0%
Initial energy: -898723.470 eV
Initial box: 30.000 Å

──────────────────────────────────────────────────
Stage 1/5: Temperature = 4000 K
──────────────────────────────────────────────────


T=4000 K:   0%|          | 0/1000 [00:00<?, ?it/s]


  Stage 1 summary:
    Energy (avg): -944636.131 eV
    Volume (avg): 26431.5 Å³
    Displacement acceptance: 659/993 = 66.4%
    Volume acceptance: 4/7 = 57.1%

──────────────────────────────────────────────────
Stage 2/5: Temperature = 3000 K
──────────────────────────────────────────────────


T=3000 K:   0%|          | 0/1000 [00:00<?, ?it/s]


  Stage 2 summary:
    Energy (avg): -17897975.163 eV
    Volume (avg): 26260.7 Å³
    Displacement acceptance: 610/991 = 61.6%
    Volume acceptance: 4/9 = 44.4%

──────────────────────────────────────────────────
Stage 3/5: Temperature = 2000 K
──────────────────────────────────────────────────


...


  Stage 3 summary:
    Energy (avg): -100020243.497 eV
    Volume (avg): 23056.1 Å³
    Displacement acceptance: 551/989 = 55.7%
    Volume acceptance: 8/11 = 72.7%

──────────────────────────────────────────────────
Stage 4/5: Temperature = 1000 K
──────────────────────────────────────────────────


T=1000 K:   0%|          | 0/1000 [00:00<?, ?it/s]


  Stage 4 summary:
    Energy (avg): -191686913.052 eV
    Volume (avg): 20960.6 Å³
    Displacement acceptance: 530/986 = 53.8%
    Volume acceptance: 10/14 = 71.4%

──────────────────────────────────────────────────
Stage 5/5: Temperature = 300 K
──────────────────────────────────────────────────


T=300 K:   0%|          | 0/1000 [00:00<?, ?it/s]


  Stage 5 summary:
    Energy (avg): -312687338383.674 eV
    Volume (avg): 19126.9 Å³
    Displacement acceptance: 538/989 = 54.4%
    Volume acceptance: 7/11 = 63.6%

ANNEALING COMPLETE
Total wall time: 208.7 seconds
Time per step: 41.74 ms
Neighbor list updates: 200

Displacement moves:
  Accepted: 2888/4948 (58.4%)
Volume moves:
  Accepted: 33/52 (63.5%)

Final configuration:
  Box length: 26.0001 Å
  Box volume: 17576.25 Å³
  Potential energy: -409108301092.610 eV
  Energy per atom: -208728725.0472 eV/atom

Saving final coordinates to final_structure.xyz...
Done!


# ============================================================
# Cell 11: Radial Distribution Function (RDF) calculation
# ============================================================

In [40]:
# ============================================================
# Cell 11: Radial Distribution Function (RDF) calculation
# ============================================================
from numba import njit
import math

@njit(parallel=True, fastmath=True)
def compute_rdf_histogram(coords, type_indices, pair_types, box_size, r_max, n_bins):
    """
    Compute RDF histogram for a given pair of atom types using PBC.
    
    Parameters
    ----------
    coords : ndarray (N,3)
    type_indices : ndarray (N,)
    pair_types : tuple (type_i, type_j)
    box_size : float
    r_max : float
    n_bins : int
    
    Returns
    -------
    hist : ndarray (n_bins,)
    """
    ti, tj = pair_types
    n_atoms = coords.shape[0]
    
    # Count atoms of each type
    mask_i = type_indices == ti
    mask_j = type_indices == tj
    n_i = np.sum(mask_i)
    n_j = np.sum(mask_j)
    
    if n_i == 0 or n_j == 0:
        return np.zeros(n_bins)
    
    # Pre-extract coordinates for efficiency
    coords_i = coords[mask_i]
    indices_i = np.where(mask_i)[0]
    coords_j = coords[mask_j] if ti != tj else coords_i
    n_i_actual = len(indices_i)
    n_j_actual = len(coords_j)
    
    hist = np.zeros(n_bins)
    bin_width = r_max / n_bins
    
    for a in prange(n_i_actual):
        idx_i = indices_i[a]
        xi, yi, zi = coords_i[a, 0], coords_i[a, 1], coords_i[a, 2]
        for b in range(n_j_actual):
            if ti == tj and a >= b:
                continue  # avoid double counting for identical types
            xj, yj, zj = coords_j[b, 0], coords_j[b, 1], coords_j[b, 2]
            dx = xi - xj
            dy = yi - yj
            dz = zi - zj
            # Minimum image
            dx = dx - box_size * round(dx / box_size)
            dy = dy - box_size * round(dy / box_size)
            dz = dz - box_size * round(dz / box_size)
            r = math.sqrt(dx*dx + dy*dy + dz*dz)
            if r < r_max:
                bin_idx = int(r / bin_width)
                if bin_idx < n_bins:
                    hist[bin_idx] += 1
    return hist

def compute_rdf(coords, type_indices, pair_elements, box_size, r_max=10.0, n_bins=100,
                density=None):
    """
    Compute normalized RDF for a pair of elements.
    
    Returns bins, rdf, and coordination number integration.
    """
    elem1, elem2 = pair_elements
    ti = type_map[elem1]
    tj = type_map[elem2]
    hist = compute_rdf_histogram(coords, type_indices, (ti, tj), box_size, r_max, n_bins)
    
    # Count atoms of each type
    n_i = np.sum(type_indices == ti)
    n_j = np.sum(type_indices == tj)
    
    if density is None:
        # Number density of the box
        rho = len(coords) / (box_size ** 3)
    else:
        rho = density  # atoms per Å³
    
    bin_edges = np.linspace(0, r_max, n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    dr = bin_edges[1] - bin_edges[0]
    
    # Volume of spherical shell for bin
    vol_shell = 4.0 * np.pi * (bin_centers ** 2) * dr
    
    # Normalization: g(r) = (V / (N_i * N_j)) * (hist / (4 pi r^2 dr))
    # For identical pair (i=j), using N_i*(N_i-1), but histogram computed without self pairs
    if ti == tj:
        n_i = max(1, n_i)
        norm = (box_size ** 3) / (n_i * (n_i - 1))  # approximate for identical
    else:
        norm = (box_size ** 3) / (n_i * n_j)
    
    rdf = (hist * norm) / vol_shell
    
    # Smooth rdf slightly (optional)
    return bin_centers, rdf

# Compute RDF for all required pairs
pairs_to_compute = [('Si', 'O'), ('Ca', 'O'), ('P', 'O'), ('O', 'O')]
rdfs = {}
for pair in pairs_to_compute:
    r_vals, gr = compute_rdf(coords, type_indices, pair, current_box, r_max=10.0, n_bins=200)
    rdfs[pair] = (r_vals, gr)
    print(f"RDF {pair[0]}-{pair[1]}: computed.")

RDF Si-O: computed.
RDF Ca-O: computed.
RDF P-O: computed.
RDF O-O: computed.


# ============================================================
# Cell 12: Coordination number analysis
# ============================================================

In [41]:
# ============================================================
# Cell 12: Coordination number analysis
# ============================================================
def coordination_number(r_vals, gr, rho_j, r_min, r_max_cut):
    """
    Integrate RDF to get coordination number up to a given radius.
    
    Parameters
    ----------
    r_vals : ndarray
        Radial distance values (bin centers).
    gr : ndarray
        Radial distribution function values.
    rho_j : float
        Number density of the neighbor type (atoms/Å³).
    r_min : float
        Lower bound of integration (typically 0).
    r_max_cut : float
        Upper bound of integration (first minimum).
    
    Returns
    -------
    cn : float
        Coordination number.
    """
    # Create mask for integration range
    mask = (r_vals >= r_min) & (r_vals <= r_max_cut)
    
    # Radial distribution function * 4πr² * ρ
    integrand = gr[mask] * 4.0 * np.pi * (r_vals[mask] ** 2) * rho_j
    
    # Use np.trapezoid (NumPy >= 2.0) or fallback to np.trapz (older versions)
    try:
        integral = np.trapezoid(integrand, r_vals[mask])
    except AttributeError:
        # Fallback for older NumPy versions
        integral = np.trapz(integrand, r_vals[mask])
    
    return integral

# Compute number densities for each element
atom_density = n_atoms / (current_box ** 3)
n_o = np.sum(type_indices == type_map['O'])
rho_O = n_o / (current_box ** 3)

# Compute coordination numbers for each pair
cn_dict = {}
cn_cutoffs = {
    ('Si', 'O'): 2.4,   # First minimum of Si-O RDF
    ('Ca', 'O'): 3.0,   # First minimum of Ca-O RDF
    ('P', 'O'): 2.4,    # First minimum of P-O RDF
    ('O', 'O'): 3.5     # First minimum of O-O RDF
}

print("=" * 50)
print("Coordination Number Analysis")
print("=" * 50)

for pair in pairs_to_compute:
    r_vals, gr = rdfs[pair]
    cutoff_cn = cn_cutoffs[pair]
    
    # For cation-O pairs, use oxygen density; for O-O, use oxygen density
    if pair[0] != 'O':
        rho_neighbor = rho_O
    else:
        rho_neighbor = rho_O
    
    cn = coordination_number(r_vals, gr, rho_neighbor, 0.0, cutoff_cn)
    cn_dict[pair] = cn
    
    print(f"  {pair[0]}-{pair[1]}: CN = {cn:.2f} (cutoff = {cutoff_cn:.1f} Å)")

print("=" * 50)

# Save to file
cn_df = pd.DataFrame([
    {'Pair': f'{pair[0]}-{pair[1]}', 'CN': f'{cn:.2f}', 'Cutoff (Å)': f'{cn_cutoffs[pair]:.1f}'}
    for pair, cn in cn_dict.items()
])
print("\nCoordination number summary:")
print(cn_df.to_string(index=False))

Coordination Number Analysis
  Si-O: CN = 5.54 (cutoff = 2.4 Å)
  Ca-O: CN = 9.22 (cutoff = 3.0 Å)
  P-O: CN = 5.29 (cutoff = 2.4 Å)
  O-O: CN = 6.54 (cutoff = 3.5 Å)

Coordination number summary:
Pair   CN Cutoff (Å)
Si-O 5.54        2.4
Ca-O 9.22        3.0
 P-O 5.29        2.4
 O-O 6.54        3.5


# ============================================================
# Cell 13: Plot generation
# ============================================================

In [42]:
# ============================================================
# Cell 14: Generate Interactive Plots & HTML Report
# ============================================================

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import matplotlib.pyplot as plt
import seaborn as sns

import base64
from io import BytesIO

import os
import time
import shutil
import zipfile

from pathlib import Path
from scipy.integrate import trapezoid

print("=" * 60)
print("GENERATING PLOTS & REPORT")
print("=" * 60)

# ================================================================
# OUTPUT DIRECTORY SETUP
# ================================================================

REPORT_DIR = Path("report")
REPORT_DIR.mkdir(exist_ok=True)

print(f"\n📁 Report directory: {REPORT_DIR.resolve()}")

# ================================================================
# PART 1: INTERACTIVE PLOTLY PLOTS
# ================================================================

# ------------------------------------------------
# 1a. Energy & Volume Combined Plot
# ------------------------------------------------

fig_combined = make_subplots(specs=[[{"secondary_y": True}]])

fig_combined.add_trace(
    go.Scatter(
        x=np.arange(1, len(energy_log)+1),
        y=energy_log,
        mode='lines',
        name='Energy',
        line=dict(color='#1f77b4', width=1.2)
    ),
    secondary_y=False
)

fig_combined.add_trace(
    go.Scatter(
        x=np.arange(1, len(volume_log)+1),
        y=volume_log,
        mode='lines',
        name='Volume',
        line=dict(color='#d62728', width=1.2)
    ),
    secondary_y=True
)

# Temperature stage shading
for idx, T in enumerate(TEMPERATURE_STAGES):

    start = idx * STEPS_PER_STAGE
    end = (idx + 1) * STEPS_PER_STAGE

    fig_combined.add_vrect(
        x0=start,
        x1=end,
        fillcolor=f"rgba({(idx*60)%255}, {(idx*80)%255}, {(idx*120)%255}, 0.08)",
        layer="below",
        annotation_text=f"{T} K",
        annotation_position="top"
    )

fig_combined.update_layout(
    title='Energy & Volume Evolution during MC Annealing',
    xaxis_title='MC Step',
    template='plotly_white',
    hovermode='x unified',
    height=500,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig_combined.update_yaxes(
    title_text="Energy (eV)",
    secondary_y=False
)

fig_combined.update_yaxes(
    title_text="Volume (Å³)",
    secondary_y=True
)

fig_combined.show()

# ------------------------------------------------
# 1b. RDF Plots
# ------------------------------------------------

fig_rdf = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f'{pair[0]}-{pair[1]}' for pair in pairs_to_compute],
    horizontal_spacing=0.1,
    vertical_spacing=0.12
)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd']

for idx, (pair, (r, gr)) in enumerate(rdfs.items()):

    row = idx // 2 + 1
    col = idx % 2 + 1

    fig_rdf.add_trace(
        go.Scatter(
            x=r,
            y=gr,
            mode='lines',
            name=f'{pair[0]}-{pair[1]}',
            line=dict(color=colors[idx], width=1.8),
            showlegend=False
        ),
        row=row,
        col=col
    )

    fig_rdf.update_xaxes(
        title_text='r (Å)',
        range=[0, 10],
        row=row,
        col=col
    )

    fig_rdf.update_yaxes(
        title_text='g(r)',
        row=row,
        col=col
    )

fig_rdf.update_layout(
    title='Radial Distribution Functions',
    template='plotly_white',
    height=600,
    margin=dict(l=50, r=30, t=60, b=50)
)

fig_rdf.show()

# ------------------------------------------------
# 1c. Acceptance Gauges
# ------------------------------------------------

acceptance_rates = {
    'Displacement': accepted_displ / max(attempts_displ, 1) * 100,
    'Volume': accepted_vol / max(attempts_vol, 1) * 100
}

fig_accept = make_subplots(
    rows=1,
    cols=2,
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}]],
    subplot_titles=[
        'Displacement Acceptance',
        'Volume Acceptance'
    ]
)

fig_accept.add_trace(
    go.Indicator(
        mode='gauge+number',
        value=acceptance_rates['Displacement'],
        number={'suffix': '%', 'font': {'size': 35}},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': '#3498db'},
            'steps': [
                {'range': [0, 30], 'color': '#fadbd8'},
                {'range': [30, 70], 'color': '#fdebd0'},
                {'range': [70, 100], 'color': '#d5f5e3'}
            ],
            'threshold': {
                'line': {'color': 'red', 'width': 2},
                'thickness': 0.8,
                'value': 50
            }
        },
        title={'text': 'Displacement'}
    ),
    row=1,
    col=1
)

fig_accept.add_trace(
    go.Indicator(
        mode='gauge+number',
        value=acceptance_rates['Volume'],
        number={'suffix': '%', 'font': {'size': 35}},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': '#e74c3c'},
            'steps': [
                {'range': [0, 5], 'color': '#fadbd8'},
                {'range': [5, 15], 'color': '#fdebd0'},
                {'range': [15, 100], 'color': '#d5f5e3'}
            ]
        },
        title={'text': 'Volume'}
    ),
    row=1,
    col=2
)

fig_accept.update_layout(
    title='Monte Carlo Move Acceptance Rates',
    template='plotly_white',
    height=350
)

fig_accept.show()

# ------------------------------------------------
# 1d. Energy Histogram + Rolling Average
# ------------------------------------------------

fig_hist_combo = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=[
        'Energy Distribution (Final Stage)',
        'Energy Evolution with Rolling Average'
    ],
    vertical_spacing=0.15,
    row_heights=[0.4, 0.6]
)

last_stage_start = (
    (len(TEMPERATURE_STAGES) - 1)
    * STEPS_PER_STAGE
)

last_stage_energy = energy_log[last_stage_start:]

mean_energy = np.mean(last_stage_energy)

# Histogram
fig_hist_combo.add_trace(
    go.Histogram(
        x=last_stage_energy,
        nbinsx=30,
        histnorm='probability density',
        marker=dict(
            color='#27ae60',
            line=dict(color='white', width=0.5)
        ),
        name='Energy Distribution',
        showlegend=False
    ),
    row=1,
    col=1
)

fig_hist_combo.add_vline(
    x=mean_energy,
    line_dash='dash',
    line_color='#e74c3c',
    line_width=2,
    annotation_text=f'Mean: {mean_energy:.2f}',
    annotation_position='top',
    row=1,
    col=1
)

# Rolling average
window = 50

rolling_energy = np.convolve(
    energy_log,
    np.ones(window)/window,
    mode='valid'
)

rolling_steps = np.arange(
    window,
    len(energy_log)+1
)

fig_hist_combo.add_trace(
    go.Scatter(
        x=np.arange(1, len(energy_log)+1),
        y=energy_log,
        mode='lines',
        line=dict(color='#bdc3c7', width=0.5),
        opacity=0.5,
        showlegend=False
    ),
    row=2,
    col=1
)

fig_hist_combo.add_trace(
    go.Scatter(
        x=rolling_steps,
        y=rolling_energy,
        mode='lines',
        line=dict(color='#e74c3c', width=2),
        showlegend=False
    ),
    row=2,
    col=1
)

# Temperature shading
for idx, T in enumerate(TEMPERATURE_STAGES):

    start = idx * STEPS_PER_STAGE
    end = (idx + 1) * STEPS_PER_STAGE

    fig_hist_combo.add_vrect(
        x0=start,
        x1=end,
        fillcolor=f"rgba({(idx*60)%255}, {(idx*80)%255}, {(idx*120)%255}, 0.08)",
        layer="below",
        annotation_text=f"{T} K",
        annotation_position="top",
        row=2,
        col=1
    )

fig_hist_combo.update_xaxes(
    title_text='Energy (eV)',
    row=1,
    col=1
)

fig_hist_combo.update_yaxes(
    title_text='Density',
    row=1,
    col=1
)

fig_hist_combo.update_xaxes(
    title_text='MC Step',
    row=2,
    col=1
)

fig_hist_combo.update_yaxes(
    title_text='Energy (eV)',
    row=2,
    col=1
)

fig_hist_combo.update_layout(
    title=f'Energy Analysis | Final Stage: T = {TEMPERATURE_STAGES[-1]} K',
    template='plotly_white',
    height=700,
    showlegend=False
)

fig_hist_combo.show()

# ================================================================
# SAVE INTERACTIVE HTML FILES
# ================================================================

print("\nSaving interactive Plotly HTML files...")

fig_combined.write_html(
    REPORT_DIR / "interactive_energy_volume.html"
)

fig_rdf.write_html(
    REPORT_DIR / "interactive_rdf.html"
)

fig_accept.write_html(
    REPORT_DIR / "interactive_acceptance.html"
)

fig_hist_combo.write_html(
    REPORT_DIR / "interactive_energy_analysis.html"
)

print("✓ Interactive plots saved!")

# ================================================================
# PART 2: STATIC PNG FOR REPORT
# ================================================================

print("\nGenerating static PNG images for report...")

# ------------------------------------------------
# Energy & Volume
# ------------------------------------------------

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(10, 6),
    sharex=True
)

steps = np.arange(1, len(energy_log)+1)

ax1.plot(
    steps,
    energy_log,
    lw=0.8,
    color='#1f77b4'
)

ax1.set_ylabel('Total Energy (eV)')
ax1.set_title('Energy Evolution during Monte Carlo Annealing')

ax2.plot(
    steps,
    volume_log,
    lw=0.8,
    color='#d62728'
)

ax2.set_ylabel('Volume (Å³)')
ax2.set_xlabel('MC Step')
ax2.set_title('Volume Evolution (NPT)')

for ax in (ax1, ax2):

    ax.grid(True, alpha=0.3)

    for idx, T in enumerate(TEMPERATURE_STAGES):

        start = idx * STEPS_PER_STAGE
        end = (idx + 1) * STEPS_PER_STAGE

        ax.axvspan(
            start,
            end,
            alpha=0.1,
            color=f'C{idx}'
        )

        ax.text(
            start + STEPS_PER_STAGE//2,
            ax.get_ylim()[1]*0.95,
            f'{T} K',
            ha='center',
            fontsize=8
        )

plt.tight_layout()

plt.savefig(
    REPORT_DIR / 'energy_volume.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("✓ energy_volume.png")

# ------------------------------------------------
# RDF
# ------------------------------------------------

fig, axes = plt.subplots(
    2,
    2,
    figsize=(10, 8)
)

axes = axes.flatten()

for idx, (pair, (r, gr)) in enumerate(rdfs.items()):

    ax = axes[idx]

    ax.plot(
        r,
        gr,
        lw=1.5,
        color=colors[idx]
    )

    ax.set_xlabel('r (Å)')
    ax.set_ylabel('g(r)')
    ax.set_title(f'{pair[0]}-{pair[1]}')
    ax.set_xlim(0, 10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()

plt.savefig(
    REPORT_DIR / 'rdf_plots.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("✓ rdf_plots.png")

# ------------------------------------------------
# Acceptance
# ------------------------------------------------

fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(
    acceptance_rates.keys(),
    acceptance_rates.values(),
    color=['#3498db', '#e74c3c']
)

ax.set_ylabel('Acceptance Rate (%)')
ax.set_title('Monte Carlo Move Acceptance')

for bar, v in zip(bars, acceptance_rates.values()):

    ax.text(
        bar.get_x() + bar.get_width()/2,
        v + 1,
        f'{v:.1f}%',
        ha='center',
        fontsize=11
    )

ax.set_ylim(
    0,
    max(100, max(acceptance_rates.values())*1.2)
)

plt.tight_layout()

plt.savefig(
    REPORT_DIR / 'acceptance.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("✓ acceptance.png")

# ------------------------------------------------
# Histogram
# ------------------------------------------------

fig, ax = plt.subplots(figsize=(6, 4))

ax.hist(
    last_stage_energy,
    bins=30,
    density=True,
    alpha=0.7,
    color='#27ae60',
    edgecolor='white'
)

ax.axvline(
    mean_energy,
    color='#e74c3c',
    linestyle='--',
    linewidth=2,
    label=f'Mean: {mean_energy:.2f} eV'
)

ax.set_xlabel('Total Energy (eV)')
ax.set_ylabel('Probability Density')

ax.set_title(
    f'Energy Distribution at T = {TEMPERATURE_STAGES[-1]} K'
)

ax.legend()

plt.tight_layout()

plt.savefig(
    REPORT_DIR / 'energy_histogram.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("✓ energy_histogram.png")

# ================================================================
# PART 3: HTML REPORT
# ================================================================

print("\nGenerating HTML report...")

def image_to_base64(filepath):

    filepath = Path(filepath)

    if filepath.exists():

        with open(filepath, "rb") as img_file:

            return base64.b64encode(
                img_file.read()
            ).decode('utf-8')

    return ""

# ------------------------------------------------
# Load images
# ------------------------------------------------

img_b64 = {}

for img_name in [
    'energy_volume',
    'rdf_plots',
    'acceptance',
    'energy_histogram'
]:

    path = REPORT_DIR / f'{img_name}.png'

    img_b64[img_name] = image_to_base64(path)

# ------------------------------------------------
# Density calculation
# ------------------------------------------------

molar_mass = {
    'Si': 28.0855,
    'Ca': 40.078,
    'P': 30.9738,
    'O': 15.999
}

n_si, n_ca, n_p, n_o = [
    atom_symbols.count(el)
    for el in ['Si', 'Ca', 'P', 'O']
]

total_mass_amu = (
    n_si*molar_mass['Si']
    + n_ca*molar_mass['Ca']
    + n_p*molar_mass['P']
    + n_o*molar_mass['O']
)

total_mass_g = total_mass_amu / NA

final_volume_cm3 = (current_box**3) * 1e-24

final_density = total_mass_g / final_volume_cm3

# ================================================================
# HTML REPORT CONTENT
# ================================================================

html_content = f"""
<!DOCTYPE html>
<html lang="en">

<head>

<meta charset="UTF-8">

<title>
MC NPT Simulation Report
</title>

<style>

body {{
    font-family: Arial, sans-serif;
    background: #f5f5f5;
    margin: 0;
    padding: 20px;
    color: #333;
}}

.container {{
    max-width: 1100px;
    margin: auto;
}}

.section {{
    background: white;
    padding: 25px;
    margin-bottom: 20px;
    border-radius: 10px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
}}

h1, h2 {{
    color: #2c3e50;
}}

table {{
    width: 100%;
    border-collapse: collapse;
    margin-top: 15px;
}}

th, td {{
    border: 1px solid #ddd;
    padding: 10px;
}}

th {{
    background: #3498db;
    color: white;
}}

tr:nth-child(even) {{
    background: #f9f9f9;
}}

img {{
    width: 100%;
    margin-top: 15px;
    border-radius: 8px;
}}

.stats {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
    gap: 15px;
}}

.card {{
    background: #ecf0f1;
    padding: 18px;
    border-radius: 8px;
    text-align: center;
}}

.value {{
    font-size: 26px;
    font-weight: bold;
}}

.label {{
    font-size: 12px;
    color: #666;
}}

</style>

</head>

<body>

<div class="container">

<div class="section">

<h1>
🔬 Monte Carlo NPT Simulation Report
</h1>

<p>
Generated on:
{time.strftime('%B %d, %Y at %H:%M')}
</p>

</div>

<div class="section">

<h2>
📊 Key Results
</h2>

<div class="stats">

<div class="card">
<div class="value">{energy_log[-1]:.2f}</div>
<div class="label">Final Energy (eV)</div>
</div>

<div class="card">
<div class="value">{current_box:.3f}</div>
<div class="label">Final Box (Å)</div>
</div>

<div class="card">
<div class="value">{final_density:.2f}</div>
<div class="label">Density (g/cm³)</div>
</div>

<div class="card">
<div class="value">{accepted_displ/max(1,attempts_displ)*100:.1f}%</div>
<div class="label">Displacement Acceptance</div>
</div>

</div>

</div>

<div class="section">

<h2>
⚙️ Simulation Parameters
</h2>

<table>

<tr>
<th>Parameter</th>
<th>Value</th>
</tr>

<tr>
<td>Atoms</td>
<td>{n_atoms}</td>
</tr>

<tr>
<td>Temperature stages</td>
<td>{' → '.join(map(str, TEMPERATURE_STAGES))} K</td>
</tr>

<tr>
<td>Total MC steps</td>
<td>{TOTAL_STEPS}</td>
</tr>

<tr>
<td>Target pressure</td>
<td>{P_TARGET} GPa</td>
</tr>

<tr>
<td>Initial Box</td>
<td>{BOX_LENGTH:.3f} Å</td>
</tr>

<tr>
<td>Final Box</td>
<td>{current_box:.3f} Å</td>
</tr>

</table>

</div>

<div class="section">

<h2>
📈 Energy & Volume Evolution
</h2>

<img src="data:image/png;base64,{img_b64['energy_volume']}">

</div>

<div class="section">

<h2>
🔍 RDF Plots
</h2>

<img src="data:image/png;base64,{img_b64['rdf_plots']}">

</div>

<div class="section">

<h2>
✅ Acceptance Rates
</h2>

<img src="data:image/png;base64,{img_b64['acceptance']}">

</div>

<div class="section">

<h2>
📊 Energy Histogram
</h2>

<img src="data:image/png;base64,{img_b64['energy_histogram']}">

</div>

<div class="section">

<h2>
🔗 Coordination Numbers
</h2>

<table>

<tr>
<th>Pair</th>
<th>CN</th>
<th>Cutoff (Å)</th>
</tr>
"""

for pair in pairs_to_compute:

    cutoff = cn_cutoffs[pair]
    cn = cn_dict.get(pair, 0)

    html_content += f"""
<tr>
<td>{pair[0]}-{pair[1]}</td>
<td>{cn:.2f}</td>
<td>{cutoff:.2f}</td>
</tr>
"""

html_content += f"""

</table>

</div>

<div class="section">

<h2>
🩺 Diagnostics
</h2>

<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Initial total energy</td>
<td>{U_total:.3f} eV</td>
</tr>

<tr>
<td>Final total energy</td>
<td>{energy_log[-1]:.3f} eV</td>
</tr>

<tr>
<td>Energy change</td>
<td>{energy_log[-1] - U_total:.3f} eV</td>
</tr>

<tr>
<td>Volume change</td>
<td>{current_box**3 - BOX_LENGTH**3:.2f} Å³</td>
</tr>

<tr>
<td>Neighbor list updates</td>
<td>{neighbor_update_counter}</td>
</tr>

<tr>
<td>Total wall time</td>
<td>{elapsed_time:.2f} s</td>
</tr>

</table>

</div>

</div>

</body>
</html>
"""

# ================================================================
# SAVE HTML REPORT
# ================================================================

report_path = REPORT_DIR / "ternary_report.html"

with open(report_path, "w", encoding="utf-8") as f:

    f.write(html_content)

print("✓ ternary_report.html")

# ================================================================
# COPY XYZ FILES
# ================================================================

extra_files = [
    "trajectory.xyz",
    "final_structure.xyz"
]

for fname in extra_files:

    src = Path(fname)

    if src.exists():

        dst = REPORT_DIR / fname

        shutil.copy2(src, dst)

        print(f"✓ {fname}")

# ================================================================
# CREATE ZIP ARCHIVE
# ================================================================

zip_path = Path("report.zip")

with zipfile.ZipFile(
    zip_path,
    'w',
    zipfile.ZIP_DEFLATED
) as zipf:

    for file in REPORT_DIR.rglob("*"):

        zipf.write(
            file,
            arcname=file.relative_to(REPORT_DIR.parent)
        )

print(f"✓ ZIP archive created: {zip_path}")

# ================================================================
# FINAL SUMMARY
# ================================================================

print("\n" + "=" * 60)
print("✅ REPORT GENERATION COMPLETE")
print("=" * 60)

print(f"\n📁 Output directory:")
print(f"   {REPORT_DIR.resolve()}")

print(f"\n📦 ZIP archive:")
print(f"   {zip_path.resolve()}")

print(f"\n📊 Quick Summary:")

print(f"   Final Energy:      {energy_log[-1]:.3f} eV")
print(f"   Final Box:         {current_box:.4f} Å")
print(f"   Final Density:     {final_density:.2f} g/cm³")

print(
    f"   Disp. Acceptance:  "
    f"{accepted_displ/max(1,attempts_displ)*100:.1f}%"
)

print(
    f"   Vol. Acceptance:   "
    f"{accepted_vol/max(1,attempts_vol)*100:.1f}%"
)

print(
    f"   Si-O CN:           "
    f"{cn_dict.get(('Si','O'), 0):.2f}"
)

print(
    f"   Ca-O CN:           "
    f"{cn_dict.get(('Ca','O'), 0):.2f}"
)

print(
    f"   P-O CN:            "
    f"{cn_dict.get(('P','O'), 0):.2f}"
)

print("=" * 60)


GENERATING PLOTS & REPORT

📁 Report directory: C:\Users\1idjl\Desktop\Work\SiO2_CaO_P2O5\report



Saving interactive Plotly HTML files...
✓ Interactive plots saved!

Generating static PNG images for report...
✓ energy_volume.png
✓ rdf_plots.png
✓ acceptance.png
✓ energy_histogram.png

Generating HTML report...
✓ ternary_report.html
✓ trajectory.xyz
✓ final_structure.xyz
✓ ZIP archive created: report.zip

✅ REPORT GENERATION COMPLETE

📁 Output directory:
   C:\Users\1idjl\Desktop\Work\SiO2_CaO_P2O5\report

📦 ZIP archive:
   C:\Users\1idjl\Desktop\Work\SiO2_CaO_P2O5\report.zip

📊 Quick Summary:
   Final Energy:      -409108301092.610 eV
   Final Box:         26.0001 Å
   Final Density:     4.09 g/cm³
   Disp. Acceptance:  58.4%
   Vol. Acceptance:   63.5%
   Si-O CN:           5.54
   Ca-O CN:           9.22
   P-O CN:            5.29
